# <font color="#418FDE" size="6.5" uppercase>**Probability Calibration**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Evaluate probabilistic predictions using calibration curves and proper scoring metrics. 
- Calibrate classifiers with sigmoid and isotonic methods using cross-validation. 
- Apply isotonic regression for monotonic calibration-style relationships. 


## **1. Calibration Metrics**

### **1.1. Probabilistic Predictions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_01.jpg?v=1784006110" width="250">



>* Probabilities show uncertainty, not just labels
>* Confidence levels guide better risk decisions

>* Probabilities should match long-run event rates
>* Calibration measures probability reliability, not accuracy

>* Calibration differs from ranking ability
>* Reliable probabilities guide better decisions



In [ ]:
#@title Python Code - Probabilistic Predictions

# This script checks whether predicted probabilities are trustworthy.
# Calibration compares predicted risk with observed event frequency.
# The plot shows perfect, calibrated, and overconfident probabilities.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

# Fixed values make the example repeatable for every run.
rng = np.random.default_rng(42)
true_probability = rng.uniform(0.02, 0.98, size=2000)
actual_outcome = rng.binomial(1, true_probability)

# One model reports honest probabilities, another exaggerates confidence.
calibrated_prediction = true_probability.copy()
overconfident_prediction = true_probability ** 2

# Proper scoring metrics reward accurate probability estimates.
calibrated_brier = brier_score_loss(actual_outcome, calibrated_prediction)
overconfident_brier = brier_score_loss(actual_outcome, overconfident_prediction)

calibrated_log = log_loss(actual_outcome, calibrated_prediction)
overconfident_log = log_loss(actual_outcome, overconfident_prediction)

# Calibration curves group similar predictions into probability bins.
calibrated_fraction, calibrated_mean = calibration_curve(
    actual_outcome, calibrated_prediction, n_bins=10, strategy="uniform"
)

over_fraction, over_mean = calibration_curve(
    actual_outcome, overconfident_prediction, n_bins=10, strategy="uniform"
)

print(f"scikit-learn version: {sklearn_version}")
print(f"Calibrated model Brier score: {calibrated_brier:.3f}")
print(f"Overconfident model Brier score: {overconfident_brier:.3f}")
print(f"Calibrated model log loss: {calibrated_log:.3f}")
print(f"Overconfident model log loss: {overconfident_log:.3f}")

# Points near the diagonal mean probabilities match observed frequencies.
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(calibrated_mean, calibrated_fraction, "o-", label="Calibrated")
ax.plot(over_mean, over_fraction, "s-", label="Overconfident")

ax.set_title("Calibration Curve for Probabilistic Predictions")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive frequency")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.legend()
plt.show()



### **1.2. Calibration Curves**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_02.jpg?v=1784006111" width="250">



>* Compare predicted probabilities with actual outcomes
>* Reveal mismatches between confidence and reality

>* Group predictions and compare actual outcome rates
>* Curve shape reveals overconfidence or underprediction

>* Check sample size and probability grouping.
>* Use representative data and supporting metrics.



In [ ]:
#@title Python Code - Calibration Curves

# This script plots a simple calibration curve.
# Calibration compares predicted probabilities with observed frequencies.
# A well calibrated model follows the diagonal.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve

# Create a small binary classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    random_state=42,
)

# Split data so calibration is checked on unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Validate that both classes appear in the test set.
if len(np.unique(y_test)) != 2:
    raise ValueError("The test set must contain both classes.")

# Fit one beginner friendly probability model.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predict probabilities for the positive class.
predicted_probability = model.predict_proba(X_test)[:, 1]

# Compute proper scoring metrics for probability quality.
brier = brier_score_loss(y_test, predicted_probability)
logloss = log_loss(y_test, predicted_probability)

# Bin predictions and compare confidence with reality.
observed_rate, mean_probability = calibration_curve(
    y_test,
    predicted_probability,
    n_bins=8,
    strategy="uniform",
)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Brier score, lower is better: {brier:.3f}")
print(f"Log loss, lower is better: {logloss:.3f}")

# Plot the calibration curve against perfect calibration.
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", label="Perfect calibration")
ax.plot(mean_probability, observed_rate, marker="o", label="Logistic model")
ax.set_title("Calibration Curve")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive frequency")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
plt.show()



### **1.3. Brier Log Loss**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_03.jpg?v=1784006108" width="250">



>* Proper scores reward honest uncertainty estimates
>* Brier score measures squared probability error

>* Log loss heavily penalizes confident wrong predictions
>* Better calibration supports safer decisions

>* Brier and log loss reveal different strengths
>* Use calibration curves to locate miscalibration



In [ ]:
#@title Python Code - Brier Log Loss

# Compare probability scores for simple forecasts.
# Brier score measures squared probability error.
# Log loss strongly punishes confident mistakes.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss
from sklearn.metrics import log_loss
import sklearn

# These outcomes represent whether an event happened.
actual_outcomes = np.array([1, 0, 1, 0, 1, 0, 1, 0])

# Three forecasters make different probability predictions.
careful_probabilities = np.array([0.70, 0.30, 0.65, 0.35, 0.75, 0.25, 0.60, 0.40])
overconfident_probabilities = np.array([0.99, 0.01, 0.95, 0.05, 0.98, 0.02, 0.01, 0.99])
underconfident_probabilities = np.array([0.55, 0.45, 0.58, 0.42, 0.60, 0.40, 0.52, 0.48])

# Store the forecasts with beginner-friendly names.
forecast_sets = {
    "Careful": careful_probabilities,
    "Overconfident": overconfident_probabilities,
    "Underconfident": underconfident_probabilities,
}

# Validate that every forecast matches the outcomes.
for name, probabilities in forecast_sets.items():
    if probabilities.shape != actual_outcomes.shape:
        raise ValueError(name + " probabilities must match the outcomes.")

# Compute both proper scoring metrics for each forecaster.
score_rows = []
for name, probabilities in forecast_sets.items():
    brier = brier_score_loss(actual_outcomes, probabilities)
    loss = log_loss(actual_outcomes, probabilities, labels=[0, 1])
    score_rows.append((name, brier, loss))

print("scikit-learn version:", sklearn.__version__)
print("Lower scores are better for both metrics.")
for name, brier, loss in score_rows:
    print(name + ": Brier=" + str(round(brier, 3)) + ", Log loss=" + str(round(loss, 3)))

# Plot the two metrics side by side for comparison.
names = [row[0] for row in score_rows]
brier_scores = [row[1] for row in score_rows]
log_losses = [row[2] for row in score_rows]

x_positions = np.arange(len(names))
bar_width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(x_positions - bar_width / 2, brier_scores, bar_width, label="Brier")
ax.bar(x_positions + bar_width / 2, log_losses, bar_width, label="Log loss")
ax.set_title("Brier Score and Log Loss Compare Probability Quality")
ax.set_xlabel("Forecast style")

ax.set_ylabel("Score, lower is better")
ax.set_xticks(x_positions)
ax.set_xticklabels(names)
ax.legend()
plt.show()



## **2. Calibration Methods**

### **2.1. Cross Validated Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_01.jpg?v=1784006114" width="250">



>* Adjust probabilities to match observed outcomes
>* Use out-of-fold predictions for fair calibration

>* Use folds to create held-out predictions
>* Fit calibrator, then refit final classifier

>* Uses limited data while reducing calibration bias
>* Still requires independent testing for trustworthy decisions



In [ ]:
#@title Python Code - Cross Validated Calibration

# This example calibrates predicted probabilities.
# Cross-validation keeps calibration training more honest.
# The plot compares uncalibrated and calibrated probabilities.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.calibration import CalibrationDisplay
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

# A small synthetic dataset keeps the lesson self-contained.
features, target = make_classification(
    n_samples=1200,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    class_sep=0.8,
    flip_y=0.08,
    random_state=42,
)

# Stratification preserves the class balance in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# This check gives a clear message if data creation changes.
if len(np.unique(y_train)) != 2:
    raise ValueError("This lesson needs exactly two classes.")

# The base model is intentionally simple for beginners.
base_model = LogisticRegression(max_iter=1000, random_state=42)
base_model.fit(X_train, y_train)
base_probabilities = base_model.predict_proba(X_test)[:, 1]

# Sigmoid calibration learns from out-of-fold training predictions.
sigmoid_model = CalibratedClassifierCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    method="sigmoid",
    cv=5,
)
sigmoid_model.fit(X_train, y_train)
sigmoid_probabilities = sigmoid_model.predict_proba(X_test)[:, 1]

# Isotonic calibration is flexible but still cross-validated here.
isotonic_model = CalibratedClassifierCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    method="isotonic",
    cv=5,
)
isotonic_model.fit(X_train, y_train)
isotonic_probabilities = isotonic_model.predict_proba(X_test)[:, 1]

# Lower Brier scores mean better probability accuracy.
base_brier = brier_score_loss(y_test, base_probabilities)
sigmoid_brier = brier_score_loss(y_test, sigmoid_probabilities)
isotonic_brier = brier_score_loss(y_test, isotonic_probabilities)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Brier score, uncalibrated: {base_brier:.3f}")
print(f"Brier score, sigmoid CV calibrated: {sigmoid_brier:.3f}")
print(f"Brier score, isotonic CV calibrated: {isotonic_brier:.3f}")

# A calibration curve shows predicted versus observed probabilities.
fig, ax = plt.subplots(figsize=(7, 5))
CalibrationDisplay.from_predictions(
    y_test,
    base_probabilities,
    n_bins=8,
    name="Uncalibrated",
    ax=ax,
)
CalibrationDisplay.from_predictions(
    y_test,
    sigmoid_probabilities,
    n_bins=8,
    name="Sigmoid CV",
    ax=ax,
)
CalibrationDisplay.from_predictions(
    y_test,
    isotonic_probabilities,
    n_bins=8,
    name="Isotonic CV",
    ax=ax,
)

ax.set_title("Cross-Validated Probability Calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive rate")
ax.legend(loc="best")
plt.show()



### **2.2. Sigmoid Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_02.jpg?v=1784006115" width="250">



>* Converts raw scores into better probabilities
>* Uses smooth curves while preserving ranking

>* Fit sigmoid using held-out model predictions
>* Convert ranking scores into reliable probabilities

>* Stable choice for limited calibration data
>* May miss complex miscalibration patterns



In [ ]:
#@title Python Code - Sigmoid Calibration

# This example demonstrates sigmoid probability calibration.
# Cross-validation learns calibration without reusing fitted scores.
# The plot compares uncalibrated and calibrated probabilities.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.calibration import CalibrationDisplay
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# A small synthetic dataset keeps the lesson reproducible.
features, target = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42,
)

# Stratification preserves the class balance in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# This check gives a clear message if data shapes disagree.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Training features and labels must have matching rows.")

# A shallow tree can rank cases but may give rough probabilities.
base_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
base_tree.fit(X_train, y_train)
raw_probabilities = base_tree.predict_proba(X_test)[:, 1]

# Sigmoid calibration fits a smooth probability mapping using folds.
calibrated_tree = CalibratedClassifierCV(
    estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
    method="sigmoid",
    cv=5,
)
calibrated_tree.fit(X_train, y_train)
calibrated_probabilities = calibrated_tree.predict_proba(X_test)[:, 1]

# Brier score is lower when probabilities match outcomes better.
raw_brier = brier_score_loss(y_test, raw_probabilities)
calibrated_brier = brier_score_loss(y_test, calibrated_probabilities)
print("scikit-learn version:", sklearn.__version__)
print("Uncalibrated Brier score:", round(raw_brier, 3))
print("Sigmoid calibrated Brier score:", round(calibrated_brier, 3))

# One calibration plot shows predicted probability versus observed frequency.
fig, ax = plt.subplots(figsize=(7, 5))
CalibrationDisplay.from_predictions(
    y_test,
    raw_probabilities,
    n_bins=8,
    name="Uncalibrated tree",
    ax=ax,
)

# The calibrated curve should move closer to the diagonal line.
CalibrationDisplay.from_predictions(
    y_test,
    calibrated_probabilities,
    n_bins=8,
    name="Sigmoid calibrated tree",
    ax=ax,
)
ax.set_title("Sigmoid Calibration With Cross-Validation")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive rate")
plt.show()



### **2.3. Isotonic Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_03.jpg?v=1784006117" width="250">



>* Maps raw scores to better probabilities
>* Preserves score order using validation data

>* Captures uneven calibration beyond sigmoid curves
>* Needs large validation data to avoid noise

>* Use held-out predictions for isotonic calibration
>* Improve probability trust in real applications



In [ ]:
#@title Python Code - Isotonic Calibration

# This example compares uncalibrated and isotonic probabilities.
# Cross-validation keeps calibration separate from model fitting.
# The plot should show improved probability reliability.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.calibration import CalibratedClassifierCV
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

# A synthetic binary dataset keeps the lesson self-contained.
features, target = make_classification(
    n_samples=3000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    random_state=42,
)

# Stratification preserves the class balance in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# This check gives a clear message if data creation changes.
if X_train.shape[0] == 0 or X_test.shape[0] == 0:
    raise ValueError("Training and test sets must both contain rows.")

# A shallow tree gives useful but imperfect probability estimates.
base_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
base_tree.fit(X_train, y_train)

# Isotonic calibration learns a monotonic probability mapping by folds.
calibrated_tree = CalibratedClassifierCV(
    estimator=DecisionTreeClassifier(max_depth=4, random_state=42),
    method="isotonic",
    cv=5,
)
calibrated_tree.fit(X_train, y_train)

# Lower Brier score means better probabilistic predictions.
raw_probabilities = base_tree.predict_proba(X_test)[:, 1]
calibrated_probabilities = calibrated_tree.predict_proba(X_test)[:, 1]
raw_brier = brier_score_loss(y_test, raw_probabilities)
calibrated_brier = brier_score_loss(y_test, calibrated_probabilities)

print("scikit-learn version:", sklearn.__version__)
print("Uncalibrated Brier score:", round(raw_brier, 4))
print("Isotonic calibrated Brier score:", round(calibrated_brier, 4))

# Calibration curves compare predicted probability with observed frequency.
fig, ax = plt.subplots(figsize=(7, 5))
CalibrationDisplay.from_predictions(
    y_test,
    raw_probabilities,
    n_bins=10,
    name="Uncalibrated tree",
    ax=ax,
)
CalibrationDisplay.from_predictions(
    y_test,
    calibrated_probabilities,
    n_bins=10,
    name="Isotonic calibrated tree",
    ax=ax,
)

ax.set_title("Isotonic calibration with cross-validation")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive frequency")
ax.legend(loc="best")
plt.show()



## **3. Advanced Calibration**

### **3.1. Frozen Estimator**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_01.jpg?v=1784006101" width="250">



>* Frozen models keep learned rankings fixed
>* Isotonic calibration adjusts probabilities monotonically

>* Keep approved base models unchanged
>* Update calibration using recent validation outcomes

>* Use independent data to avoid overconfident calibration
>* Isotonic needs enough examples for stable mapping



In [ ]:
#@title Python Code - Frozen Estimator

# This example freezes a fitted classifier.
# Isotonic calibration learns a monotonic probability map.
# The plot compares raw and calibrated probabilities.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    FrozenEstimator = None

# A small synthetic dataset keeps the lesson reproducible.
features, target = make_classification(
    n_samples=3000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    class_sep=0.8,
    flip_y=0.08,
    random_state=42,
)

# Separate training, calibration, and test data prevents leakage.
train_features, temp_features, train_target, temp_target = train_test_split(
    features,
    target,
    test_size=0.5,
    stratify=target,
    random_state=42,
)

cal_features, test_features, cal_target, test_target = train_test_split(
    temp_features,
    temp_target,
    test_size=0.5,
    stratify=temp_target,
    random_state=42,
)

# The base classifier is fitted once, then treated as frozen.
base_model = LogisticRegression(max_iter=500, random_state=42)
base_model.fit(train_features, train_target)

# Isotonic calibration learns only from held-out calibration data.
if FrozenEstimator is not None:
    calibrated_model = CalibratedClassifierCV(
        estimator=FrozenEstimator(base_model),
        method="isotonic",
    )
else:
    calibrated_model = CalibratedClassifierCV(
        estimator=base_model,
        method="isotonic",
        cv="prefit",
    )

calibrated_model.fit(cal_features, cal_target)

# Test probabilities show whether calibration improved reliability.
raw_probabilities = base_model.predict_proba(test_features)[:, 1]
calibrated_probabilities = calibrated_model.predict_proba(test_features)[:, 1]

if len(raw_probabilities) != len(test_target):
    raise ValueError("Probability and target lengths must match.")

raw_brier = brier_score_loss(test_target, raw_probabilities)
calibrated_brier = brier_score_loss(test_target, calibrated_probabilities)

print("scikit-learn version:", sklearn.__version__)
print("Raw Brier score:", round(raw_brier, 4))
print("Calibrated Brier score:", round(calibrated_brier, 4))

# Calibration curves compare predicted probability with observed frequency.
raw_fraction, raw_mean = calibration_curve(
    test_target,
    raw_probabilities,
    n_bins=10,
    strategy="uniform",
)

cal_fraction, cal_mean = calibration_curve(
    test_target,
    calibrated_probabilities,
    n_bins=10,
    strategy="uniform",
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(raw_mean, raw_fraction, "o-", label="Frozen model raw")
ax.plot(cal_mean, cal_fraction, "s-", label="Isotonic calibrated")

ax.set_title("Frozen Estimator With Isotonic Calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive frequency")
ax.legend()
plt.show()



### **3.2. Multiclass Probability Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_02.jpg?v=1784006106" width="250">



>* Calibration extends to multiple possible classes
>* Probabilities should reflect real outcome frequencies

>* Calibrate each class one-versus-rest
>* Normalize probabilities into one multiclass distribution

>* Validate isotonic calibration to avoid overfitting
>* Check all classes and secondary probabilities



In [ ]:
#@title Python Code - Multiclass Probability Calibration

# This example calibrates multiclass probability estimates.
# Isotonic calibration learns monotonic classwise mappings.
# The plot compares confidence before and after calibration.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

# Load a small packaged multiclass dataset.
wine = load_wine()
features = wine.data
target = wine.target

# Check that the lesson really has multiple classes.
class_count = len(np.unique(target))
if class_count < 3:
    raise ValueError("This example needs at least three classes.")

# Split once for training and once for final testing.
train_features, test_features, train_target, test_target = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

# Use one simple probabilistic classifier.
base_model = RandomForestClassifier(
    n_estimators=80, max_depth=3, random_state=42
)
base_model.fit(train_features, train_target)

# Calibrate each class with cross-validated isotonic regression.
calibrated_model = CalibratedClassifierCV(
    base_model, method="isotonic", cv=3
)
calibrated_model.fit(train_features, train_target)

# Compare full probability distributions on unseen test data.
raw_probabilities = base_model.predict_proba(test_features)
calibrated_probabilities = calibrated_model.predict_proba(test_features)

raw_loss = log_loss(test_target, raw_probabilities)
calibrated_loss = log_loss(test_target, calibrated_probabilities)

# Summarize the proper scoring metric for beginners.
print("scikit-learn version:", sklearn.__version__)
print("Raw multiclass log loss:", round(raw_loss, 3))
print("Isotonic calibrated log loss:", round(calibrated_loss, 3))

# Bin top-class confidence to show calibration visually.
bins = np.linspace(0.0, 1.0, 6)
raw_confidence = np.max(raw_probabilities, axis=1)
calibrated_confidence = np.max(calibrated_probabilities, axis=1)
raw_correct = base_model.predict(test_features) == test_target
calibrated_correct = calibrated_model.predict(test_features) == test_target

raw_bin_centers = []
raw_bin_accuracy = []
calibrated_bin_centers = []
calibrated_bin_accuracy = []

# Average accuracy inside each confidence bin.
for left, right in zip(bins[:-1], bins[1:]):
    raw_mask = (raw_confidence >= left) & (raw_confidence < right)
    calibrated_mask = (calibrated_confidence >= left) & (calibrated_confidence < right)

    if np.sum(raw_mask) > 0:
        raw_bin_centers.append(float(np.mean(raw_confidence[raw_mask])))
        raw_bin_accuracy.append(float(np.mean(raw_correct[raw_mask])))

    if np.sum(calibrated_mask) > 0:
        calibrated_bin_centers.append(float(np.mean(calibrated_confidence[calibrated_mask])))
        calibrated_bin_accuracy.append(float(np.mean(calibrated_correct[calibrated_mask])))

# Plot confidence against observed accuracy.
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(raw_bin_centers, raw_bin_accuracy, "o-", label="Raw model")
ax.plot(calibrated_bin_centers, calibrated_bin_accuracy, "s-", label="Isotonic calibrated")

ax.set_title("Multiclass Top-Confidence Calibration")
ax.set_xlabel("Mean predicted confidence")
ax.set_ylabel("Observed accuracy")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.legend()
plt.show()



### **3.3. Monotonic Probability Mapping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_03.jpg?v=1784006103" width="250">



>* Calibrate probabilities while preserving score order
>* Isotonic regression learns flexible monotonic mappings

>* Handles irregular but ordered score relationships
>* Corrects locally while preserving probability order

>* Balance isotonic flexibility with overfitting risk
>* Validate calibration for reliable decision support



In [ ]:
#@title Python Code - Monotonic Probability Mapping

# This example maps raw scores to calibrated probabilities.
# Isotonic regression preserves score order while adjusting probabilities.
# The plot shows a nondecreasing learned probability mapping.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
import sklearn

# These raw scores mimic an imperfect classifier output.
raw_scores = np.array([
    0.05, 0.10, 0.15, 0.20, 0.25, 0.30,
    0.35, 0.40, 0.45, 0.50, 0.55, 0.60,
    0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
])

# These observed rates are noisy but generally increase.
observed_rates = np.array([
    0.02, 0.04, 0.03, 0.10, 0.12, 0.11,
    0.20, 0.24, 0.23, 0.35, 0.38, 0.36,
    0.50, 0.58, 0.55, 0.70, 0.78, 0.76,
])

# Validate that the paired calibration data have matching lengths.
if raw_scores.shape != observed_rates.shape:
    raise ValueError("Scores and observed rates must have matching shapes.")

# Fit a flexible nondecreasing mapping from scores to probabilities.
isotonic = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
isotonic.fit(raw_scores, observed_rates)

# Predict calibrated probabilities on a smooth score grid.
score_grid = np.linspace(0.0, 1.0, 101)
calibrated_grid = isotonic.predict(score_grid)

# Check the monotonic property learned by isotonic regression.
monotonic_steps = np.diff(calibrated_grid) >= -1e-12
is_monotonic = bool(np.all(monotonic_steps))

print("scikit-learn version:", sklearn.__version__)
print("Mapping is nondecreasing:", is_monotonic)
print("Raw score 0.80 maps to:", round(float(isotonic.predict([0.80])[0]), 3))

# Plot raw calibration points and the learned monotonic mapping.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(raw_scores, observed_rates, label="Observed calibration data")
ax.plot(score_grid, calibrated_grid, color="orange", label="Isotonic mapping")

ax.set_title("Monotonic Probability Mapping with Isotonic Regression")
ax.set_xlabel("Raw model score")
ax.set_ylabel("Calibrated probability")
ax.set_ylim(-0.02, 1.02)
ax.legend()

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Probability Calibration**</font>


In this lecture, you learned to:
- Evaluate probabilistic predictions using calibration curves and proper scoring metrics. 
- Calibrate classifiers with sigmoid and isotonic methods using cross-validation. 
- Apply isotonic regression for monotonic calibration-style relationships. 

In the next Module (Module 16), we will go over 'Decision Trees'